In [3]:
import torch
import os
from pathlib import Path
import sys
sys.path.append('..')

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {DEVICE}")

DATA_DIR   = Path('../data/raw/Kaggle_Dataset/train')
VAL_DIR    = Path('../data/raw/Kaggle_Dataset/valid')
TEST_DIR   = Path('../data/raw/Kaggle_Dataset/test')

print(f"Train exists: {DATA_DIR.exists()}")
print(f"Val exists  : {VAL_DIR.exists()}")
print(f"Test exists  : {TEST_DIR.exists()}")

classes = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
print(f"\nClasses: {len(classes)}")
print(classes)

Device: mps
Train exists: True
Val exists  : True
Test exists  : True

Classes: 38
['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septor

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from torchvision.models import efficientnet_b0

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Transforms ready")

Matplotlib is building the font cache; this may take a moment.


Transforms ready


In [7]:
train_dataset = datasets.ImageFolder(root=DATA_DIR, transform=train_transforms)
val_dataset   = datasets.ImageFolder(root=VAL_DIR,  transform=val_transforms)

classes = train_dataset.classes
print(f"Classes: {len(classes)}")
print(f"Train: {len(train_dataset)}")
print(f"Val  : {len(val_dataset)}")

Classes: 38
Train: 70295
Val  : 17572


In [9]:
from src.dataset import get_dataloaders
from src.model import get_model

model_1 = get_model(num_classes =38)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /Users/prempatel/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20.5M/20.5M [00:00<00:00, 23.8MB/s]


In [10]:
model_1 = model_1.to(DEVICE)

In [11]:
print(model_1.parameters())

<generator object Module.parameters at 0x142103ae0>


In [17]:
class_counts = {cls: len(list((DATA_DIR / cls).glob('*.JPG'))) for cls in classes}
total = sum(class_counts.values())
weights = [total / (len(classes) * class_counts[cls]) for cls in classes]
weights_tensor = torch.tensor(weights, dtype=torch.float)
print(weights_tensor)

tensor([ 0.8857,  0.8986,  1.0145,  0.8892,  0.9832,  1.0609,  0.9778,  1.2495,
         0.9363,  1.0672, 63.7669,  0.9457,  0.9299,  1.0369,  1.0552,  0.8883,
         0.9714,  1.0333,  0.9333,  0.8986,  0.9208,  0.9208,  0.9789,  1.0025,
         0.8830,  1.0399,  1.0070,  0.9789,  1.0490,  0.9299,  1.0472,  0.9487,
         1.0232,  1.0255,  0.9773,  0.9105,  0.9975,  0.9275])


In [30]:
weights_tensor = weights_tensor.to(DEVICE)
criterion = nn.CrossEntropyLoss(weight = weights_tensor)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model_1.parameters()), lr=0.001)

In [31]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=0)



In [32]:
print(f"Train:{train_loader}")
print(f"val:{val_loader}")
print(f"train_length:{len(train_loader)}")
print(f"val_length:{len(val_loader)}")

Train:<torch.utils.data.dataloader.DataLoader object at 0x140ce3340>
val:<torch.utils.data.dataloader.DataLoader object at 0x140ce3880>
train_length:2197
val_length:550


In [33]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    #Training
    model.train()

    #loss calculation metric
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:        # loop ALL batches
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()        # += not =+
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total * 100
    return avg_loss, accuracy


    

In [34]:
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            
            val_output = model(images)
            loss = criterion(val_output, labels)
            
            total_loss += loss.item()
            
            val_preds = torch.argmax(val_output, dim=1)
            correct += (val_preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total * 100
    return avg_loss, accuracy

In [ ]:
EPOCHS = 10
best_val_acc = 0.0

train_losses = [] 
val_losses = []
train_accuracies = []
val_accuracies = []


for epoch in range(EPOCHS):
    t_loss, t_acc = train_one_epoch(model_1, train_loader, criterion, optimizer, DEVICE)

    v_loss, v_acc = validate(model_1, val_loader, criterion, DEVICE)

    train_losses.append(t_loss)
    val_losses.append(v_loss)
    train_accuracies.append(t_acc)
    val_accuracies.append(v_acc)

    print(f"Epoch {epoch+1:02d}/10 | Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.2f}% | Val Loss: {v_loss:.4f} | Val Acc: {v_acc:.2f}%")

if v_acc > best_val_acc:
    best_val_acc = v_acc
    torch.save(model_0.state_dict(), '../models/best_model_v2.pth')
    print(f"  → Best model saved!")

Epoch 01/10 | Train Loss: 0.5435 | Train Acc: 79.97% | Val Loss: 0.3610 | Val Acc: 89.88%
